# M06-02 - Caché y particionado


## Paso 1 Un fact más largo

Se generan 8 copias para notar en el cache local.


In [5]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, lit

spark = get_spark("novashop-m06")
base = spark.read.parquet(str(STAGING / "fact_lines"))
xl = base.withColumn("_copy", lit(-1))
for i in range(7):
    xl = xl.unionByName(base.withColumn("_copy", lit(i)))
print("particiones iniciales", xl.rdd.getNumPartitions())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
particiones iniciales 8


## Paso 2 -- Dos counts sin cache

In [6]:
import time

def timed_count(df, label):
    t0 = time.perf_counter()
    n = df.where(col("is_billable")).count()
    print(label, n, f"{time.perf_counter() - t0:.2f}s")

timed_count(xl, "1er count frío")
timed_count(xl, "2º count frío")

1er count frío 9016 0.31s
2º count frío 9016 0.25s


## Paso 3 -- caché materializado

In [7]:
warm = xl.where(col("is_billable")).cache()
timed_count(warm, "calentamiento (materializa cache)")
timed_count(warm, "caliente")

26/09/29 15:24:04 WARN CacheManager: Asked to cache already cached data.


calentamiento (materializa cache) 9016 0.31s
caliente 9016 0.24s


## Paso 4 -- Repartición por mes

In [8]:
by_month = warm.repartition(12, col("order_month"))
print("particiones", by_month.rdd.getNumPartitions())
by_month.groupBy("order_month").count().orderBy("order_month").show()


particiones 12
+-----------+-----+
|order_month|count|
+-----------+-----+
|    2024-01|  544|
|    2024-02|  648|
|    2024-03|  792|
|    2024-04|  688|
|    2024-05|  816|
|    2024-06|  792|
|    2024-07|  624|
|    2024-08|  928|
|    2024-09|  848|
|    2024-10|  928|
|    2024-11|  984|
|    2024-12|  424|
+-----------+-----+



In [ ]:
by_month.cache()


DataFrame[order_id: string, product_id: string, qty: int, unit_price: decimal(10,2), discount: double, customer_id: string, status: string, channel: string, order_ts: timestamp, gmv_line: double, order_month: string, channel_norm: string, is_billable: boolean, _copy: int]

## Mejora


In [23]:
by_month.repartition(1)
by_month.coalesce(1)

## display(by_month.coalesce(1))
##by_month.coalesce(1).show().limit(4)



DataFrame[order_id: string, product_id: string, qty: int, unit_price: decimal(10,2), discount: double, customer_id: string, status: string, channel: string, order_ts: timestamp, gmv_line: double, order_month: string, channel_norm: string, is_billable: boolean, _copy: int]